# M29 Catalog-Driven Research Workflow

This notebook uses the unified research catalog to discover existing runs, inspect lineage, validate catalog integrity, and collect relative artifact paths for follow-up analysis. It is read-only and does not run research workflows.

## 1. Build the Catalog

In [ ]:
from pathlib import Path

from src.catalog import (
    CatalogQuery,
    build_catalog,
    get_downstream_records,
    get_upstream_records,
    query_catalog,
    records_to_dicts,
    records_to_rows,
    validate_catalog,
)

In [ ]:
repo_root = "."
artifacts_root = "artifacts"

records = build_catalog(artifacts_root, repo_root=repo_root)
{
    "catalog_records": len(records),
    "artifact_state": "available" if Path(artifacts_root).exists() else "missing",
}

## 2. Query Completed Research Runs

In [ ]:
completed_strategies = query_catalog(
    records,
    CatalogQuery(run_types=("strategy",), statuses=("completed",)),
)
completed_portfolios = query_catalog(
    records,
    CatalogQuery(run_types=("portfolio",), statuses=("completed",)),
)

{
    "strategy_count": len(completed_strategies),
    "portfolio_count": len(completed_portfolios),
    "strategy_rows": records_to_rows(completed_strategies[:5]),
    "portfolio_rows": records_to_rows(completed_portfolios[:5]),
}

## 3. Filter by Metrics and Run Type

In [ ]:
completed_runs = query_catalog(
    records,
    CatalogQuery(
        run_types=("strategy", "portfolio", "alpha_evaluation"),
        statuses=("completed",),
    ),
)
metric_names = sorted(
    {
        name
        for record in completed_runs
        for name, value in (record.metrics_summary or {}).items()
        if isinstance(value, (int, float)) and not isinstance(value, bool)
    }
)

if metric_names:
    metric_name = metric_names[0]
    threshold = min(
        float((record.metrics_summary or {})[metric_name])
        for record in completed_runs
        if isinstance((record.metrics_summary or {}).get(metric_name), (int, float))
        and not isinstance((record.metrics_summary or {}).get(metric_name), bool)
    )
    metric_filtered = query_catalog(completed_runs, CatalogQuery(min_metric=(metric_name, threshold)))
else:
    metric_name = None
    metric_filtered = []

{
    "metric": metric_name,
    "matches": len(metric_filtered),
    "rows": records_to_rows(metric_filtered[:5]),
}

## 4. Inspect Lineage

In [ ]:
target = next(iter(completed_portfolios or completed_strategies or records), None)

if target is None:
    lineage_view = {
        "target": None,
        "upstream": [],
        "downstream": [],
    }
else:
    upstream = get_upstream_records(target, records, repo_root=repo_root)
    downstream = get_downstream_records(target, records, repo_root=repo_root)
    lineage_view = {
        "target": target.run_id or target.catalog_id,
        "upstream": records_to_rows(upstream),
        "downstream": records_to_rows(downstream),
    }

lineage_view

## 5. Validate Catalog Integrity

In [ ]:
validation_report = validate_catalog(records, repo_root=repo_root)
{
    "records": validation_report.total_records,
    "artifacts": validation_report.total_artifacts,
    "errors": validation_report.error_count,
    "warnings": validation_report.warning_count,
    "by_code": validation_report.summary.get("by_code", {}),
}

## 6. Load Artifact Paths for Follow-Up Analysis

In [ ]:
catalog_rows = records_to_rows(records)
artifact_roots = [
    row["artifact_root"]
    for row in catalog_rows
    if isinstance(row.get("artifact_root"), str) and row["artifact_root"]
]

{
    "artifact_roots": artifact_roots[:5],
    "record_dict_sample": records_to_dicts(records[:1]),
}

## 7. Notes on Read-Only Workflow

This notebook builds an in-memory catalog, queries records, inspects lineage, validates catalog integrity, and returns relative artifact paths. It does not execute strategies, alphas, portfolios, pipelines, campaigns, benchmark packs, validations, or repair tasks. Keep committed notebook outputs empty and treat any local exports as disposable user artifacts.